# 01 — Frame the problem

**How long will this trip take?** (Team B: regression, predicted at pickup,
target is minutes from pickup to drop-off.)

Week 1. This notebook settles three things and stops there:

1. the question, and who acts on the answer,
2. the target, and every decision hidden inside it,
3. what is knowable at the moment of prediction, and the baseline ladder.

It does **not** clean the data, split it, or fit anything. Those are weeks 2
and 3. The counts below are evidence for the decisions written in
[`PROBLEM.md`](../PROBLEM.md), not a cleaning step.

## The question

A passenger has just got into a yellow taxi and wants to know when they will
arrive. The app shows that time the moment the trip starts.

Arriving **later** than shown costs the passenger whatever they planned
around it. Arriving **earlier** costs them a wait. The two are not equal, so
the app will show the estimate plus a buffer, and sizing that buffer is part
of what this project has to deliver.

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import DATA_RAW

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

trips = pd.read_parquet(DATA_RAW / "yellow_tripdata_2026_sample.parquet")
print(f"{len(trips):,} rows x {trips.shape[1]} columns")
print(trips["tpep_pickup_datetime"].dt.to_period("M").value_counts().sort_index().to_string())

## The target

`duration_min` = `tpep_dropoff_datetime` − `tpep_pickup_datetime`, in minutes.

Both timestamps come from the meter. Neither is a feature: the drop-off time
**is** the answer.

In [ ]:
duration_min = (
    trips["tpep_dropoff_datetime"] - trips["tpep_pickup_datetime"]
).dt.total_seconds() / 60

print(duration_min.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(duration_min.clip(-10, 120), bins=130)
axes[0].set(title="Raw duration, clipped to -10..120", xlabel="minutes", ylabel="trips")
axes[1].hist(duration_min[duration_min.between(0, 60)], bins=60)
axes[1].set(title="The part that looks like trips", xlabel="minutes")
plt.show()

### The decisions inside the target

Zero durations, negative ones, trips of a few seconds and trips of several
hours all exist in this file. Each is a judgement, not a fact. The counts
below are what each candidate rule would cost — they are the evidence for the
table in `PROBLEM.md`.

Week 2 is where these rules get argued for properly and implemented in
`src.data.clean`. Nothing is dropped here.

In [ ]:
distance = trips["trip_distance"]
with np.errstate(divide="ignore", invalid="ignore"):
    speed_mph = distance / (duration_min / 60)

candidate_rules = {
    "duration <= 0":          duration_min <= 0,
    "duration < 1 min":       duration_min < 1,
    "duration > 180 min":     duration_min > 180,
    "distance == 0":          distance == 0,
    "distance > 100 mi":      distance > 100,
    "implied speed < 1 mph":  speed_mph < 1,
    "implied speed > 80 mph": speed_mph > 80,
}

n = len(trips)
summary = pd.DataFrame(
    [{"rule": name, "rows": int(mask.sum()), "pct": round(100 * mask.sum() / n, 3)}
     for name, mask in candidate_rules.items()]
)
any_rule = np.logical_or.reduce([m.to_numpy() for m in candidate_rules.values()])
summary.loc[len(summary)] = ["ANY of the above", int(any_rule.sum()), round(100 * any_rule.sum() / n, 3)]
summary

In [ ]:
# What is actually in there, so the decision is made with eyes open.
print(f"negative durations: {int((duration_min < 0).sum())}")
print(f"exactly zero:       {int((duration_min == 0).sum()):,}  "
      "— the meter started and stopped, not a trip")

print("\nzero-distance trips that still ran over a minute:")
zero_dist = (distance == 0) & (duration_min > 1)
print(f"{int(zero_dist.sum()):,} trips; median duration "
      f"{duration_min[zero_dist].median():.1f} min — the meter ran but measured nothing")

print("\nthe longest trips in the file (minutes):")
print(duration_min.nlargest(5).round(1).to_string())

print("\nthe longest distances in the file (miles):")
print(distance.nlargest(5).round(1).to_string())

## The moment of prediction

The passenger is in the car and the meter has just started.

**Knowable now:** the pickup timestamp (and the hour, day of week, month
derived from it), `PULocationID` and `DOLocationID` with their zone, borough
and airport status, `passenger_count` (entered by the driver at the start, and
absent on flex fare trips), and `trip_distance` — see the assumption below.

**Not knowable, therefore never a feature:** `tpep_dropoff_datetime`,
`fare_amount`, `tip_amount`, `total_amount`, `extra`, `mta_tax`,
`tolls_amount`, `improvement_surcharge`, `congestion_surcharge`,
`Airport_fee`, `cbd_congestion_fee`, `payment_type`, and `RatecodeID` — the
rate is set at the *end* of the trip.

### The assumption on `trip_distance`

`trip_distance` is the distance the meter measured **during** the trip, so it
already contains any detour that happened. This project treats it as the
length of the planned route. That is generous, and it flatters every score
this repository will report; a deployed app would use a routing service's
estimate and do worse. Said here, in `PROBLEM.md`, and in the model card.

In [ ]:
# Why the fare cannot be used: the meter charges for time, so it is the answer.
loose = duration_min.between(1, 180) & (distance > 0)
tight = loose & (distance <= 100) & speed_mph.between(1, 80)

for label, mask in [("duration and distance filtered only", loose),
                    ("plus the speed and distance caps", tight)]:
    frame = pd.DataFrame({
        "duration_min": duration_min[mask],
        "fare_amount": trips.loc[mask, "fare_amount"],
        "total_amount": trips.loc[mask, "total_amount"],
        "trip_distance": distance[mask],
    })
    corr = frame.corr()["duration_min"].drop("duration_min")
    print(f"{label}  (n = {int(mask.sum()):,})")
    print(corr.sort_values(ascending=False).round(3).to_string())
    print()

The fare is the trap this question comes with: the meter charges for time, so `fare_amount` carries the answer. The second trap is quieter — a handful of meter-fault distances are enough to flatten the correlation between distance and duration to almost nothing. Which rows are excluded changes what the data appears to say, before any model is fitted. That is the argument for making those rules explicit and counting them, which is week 2's job.

## The metric

- **MAE**, in minutes — the headline. It is what the passenger feels, and it
  is not dominated by the long tail.
- **RMSE** alongside it, which *is* dominated by the tail, and so says
  something the MAE hides.
- On July, once, with a bootstrap interval next to the baseline.

## The baseline ladder

1. **The median duration of May**, for every trip. The number to beat before
   anything else is worth discussing.
2. **A rule**: `trip_distance` ÷ a typical speed, the speed taken from May.
3. **A decision tree two levels deep.**

Then linear regression and a single decision tree, judged on June.

Each rung is fitted on May and scored on June, and every rung has to beat the
one below it. Scoring them needs the cleaning and the split, so the ladder is
**specified here and scored in week 2**, once the rules above are settled.

## What this notebook does not do

- No rows dropped, no cleaning function — week 2.
- No split, no features, no model fitted — weeks 2 and 3.
- **July is not opened.** It is scored once, in `04-evaluate.ipynb`.